# 15 — Engineer v2.2 Progression Features

**LeukemiPrediag v2.2 — label-free feature engineering**

Run this after notebooks **12** and **13**. Notebook 14 is analysis-only and is optional for this step.

This notebook does **not** create leukemia labels and does not train a progression classifier. It converts the existing longitudinal VAF + candidate-clone outputs into a richer patient-level feature matrix for later outcome-linked validation.

Feature families:

- VAF burden / observed thresholds,
- first-order clone kinetics,
- positive-growth doubling time,
- experimental second-order acceleration (only for trajectories with >=4 timepoints),
- posterior growth / uncertainty,
- clone diversity and dominance,
- longitudinal co-mutation architecture,
- configurable gene-presence indicators.

**Research use only. Not a diagnostic device.**

In [ ]:
# 0. Colab + project setup
from pathlib import Path
import os, sys, json
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MYDRIVE = Path("/content/drive/MyDrive")
    candidates = [
        MYDRIVE / "leukemia-prediagnose" / "blood-cancer-early-detection",
        MYDRIVE / "blood-cancer-early-detection",
    ]
    candidates += list(MYDRIVE.glob("*/blood-cancer-early-detection"))
    PROJECT_ROOT = next((p for p in candidates if p.exists()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Could not find blood-cancer-early-detection under MyDrive")
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("progression_features.py:", (SRC / "progression_features.py").exists())


In [ ]:
# 1. Import v2.2 feature engine
from progression_features import (
    ProgressionFeatureConfig,
    patient_progression_features,
    feature_manifest,
)

print("✓ progression_features imported")


In [ ]:
# 2. Paths — matched to notebooks 12/13
V2_DIR = PROJECT_ROOT / "data" / "processed" / "v2"
CLONE_DIR = V2_DIR / "clone_inference"
OUTDIR = V2_DIR / "progression_features"
OUTDIR.mkdir(parents=True, exist_ok=True)

paths = {
    "serial": V2_DIR / "real_serial_filtered.csv",
    "trajectory": CLONE_DIR / "trajectory_posteriors.csv",
    "pairs": CLONE_DIR / "pair_support.csv",
    "assignments": CLONE_DIR / "clone_assignments.csv",
    "clones": CLONE_DIR / "clone_posteriors.csv",
    "patient_clone": CLONE_DIR / "patient_clone_features.csv",
}

for name, path in paths.items():
    print(f"{name:15s}", "OK" if path.exists() else "MISSING", path)

missing = [str(p) for p in paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Missing notebook-12/13 outputs:\n" + "\n".join(missing))


In [ ]:
# 3. Load real Stage-B assets
serial = pd.read_csv(paths["serial"])
trajectory = pd.read_csv(paths["trajectory"])
pairs = pd.read_csv(paths["pairs"])
assignments = pd.read_csv(paths["assignments"])
clones = pd.read_csv(paths["clones"])
patient_clone = pd.read_csv(paths["patient_clone"])

print("serial:", serial.shape)
print("trajectory posteriors:", trajectory.shape)
print("pair support:", pairs.shape)
print("clone assignments:", assignments.shape)
print("clone posteriors:", clones.shape)
print("patient clone features:", patient_clone.shape)


## 4. Build v2.2 features

The support threshold remains **0.80**, matching notebook 13. VAF thresholds (10% and 20%) are stored as covariates only — they are **not** treated as standalone diagnostic rules.

In [ ]:
cfg = ProgressionFeatureConfig(
    pair_support_threshold=0.80,
    vaf_thresholds=(0.10, 0.20),
    min_acceleration_points=4,
)

patient_features, variant_kinetics, supported_pairs = patient_progression_features(
    serial,
    trajectory_posteriors=trajectory,
    pair_support=pairs,
    assignments=assignments,
    clone_posteriors=clones,
    patient_clone_features=patient_clone,
    cfg=cfg,
)

print("patient_features:", patient_features.shape)
print("variant_kinetics:", variant_kinetics.shape)
print("supported_pairs:", supported_pairs.shape)


In [ ]:
# 5. Sanity checks
assert patient_features["patient_id"].is_unique
assert len(patient_features) == serial["patient_id"].nunique()
assert len(variant_kinetics) == serial.groupby(["patient_id", "variant_id"]).ngroups
numeric_matrix = patient_features.select_dtypes(include=[np.number]).to_numpy(dtype=float)
assert not np.isinf(numeric_matrix).any(), "Infinite numeric feature detected"

summary = {
    "patients": int(len(patient_features)),
    "variant_trajectories": int(len(variant_kinetics)),
    "supported_variant_pairs": int(len(supported_pairs)),
    "patients_with_acceleration_estimable": int((patient_features["n_acceleration_estimable"] > 0).sum()),
    "patients_ever_vaf_ge_20pct": int(patient_features["any_vaf_ge_20pct"].sum()),
    "patients_with_supported_clone_structure": int(patient_features["structure_supported"].fillna(0).sum()),
}
print(json.dumps(summary, indent=2))


## 6. Inspect the new kinetics

`doubling_time` is only defined for positive estimated log-VAF growth. `acceleration` is exploratory and is only estimated with at least four unique timepoints.

In [ ]:
cols = [
    "patient_id", "cohort", "n_variants_observed", "max_vaf_observed",
    "posterior_max_growth", "fastest_posterior_doubling_time_years",
    "n_acceleration_estimable", "max_log_vaf_acceleration",
    "n_supported_variant_pairs", "n_supported_gene_pairs",
    "dominant_clone_fraction_proxy", "clone_vaf_entropy",
    "any_vaf_ge_20pct",
]
display(patient_features[[c for c in cols if c in patient_features.columns]].head(20))


In [ ]:
# 7. Top fastest positive-growth trajectories (descriptive only)
v = variant_kinetics.copy()
v = v[np.isfinite(v["observed_doubling_time_years"])].sort_values("observed_doubling_time_years")
display(v[[
    "patient_id", "cohort", "gene", "protein_change", "variant_id",
    "max_vaf_observed", "observed_log_vaf_growth",
    "observed_doubling_time_years", "log_vaf_acceleration",
]].head(25))


In [ ]:
# 8. Supported longitudinal co-mutation pairs
if len(supported_pairs):
    show = [
        "patient_id", "cohort", "gene_a", "protein_a", "gene_b", "protein_b",
        "support", "p_corr", "p_ratio_stable", "p_slope_similar", "gene_pair",
    ]
    display(supported_pairs[[c for c in show if c in supported_pairs.columns]]
            .sort_values("support", ascending=False).head(30))
else:
    print("No supported pairs at the configured threshold.")


In [ ]:
# 9. Save v2.2 research assets
patient_features.to_csv(OUTDIR / "patient_progression_features.csv", index=False)
variant_kinetics.to_csv(OUTDIR / "variant_kinetics.csv", index=False)
supported_pairs.to_csv(OUTDIR / "supported_comutation_pairs.csv", index=False)

manifest = feature_manifest(patient_features, cfg)
manifest["real_data_audit"] = summary
with open(OUTDIR / "feature_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved to:", OUTDIR)
for p in sorted(OUTDIR.iterdir()):
    print(" -", p.name)


## Interpretation boundary

These v2.2 features are intended to make future progression modeling stronger, but the current public cohorts still do not provide a clean future-leukemia endpoint.

Do **not** interpret:

- VAF >= 20%,
- short doubling time,
- positive acceleration,
- a supported co-mutation pair, or
- a multi-variant candidate clone

as a standalone probability of leukemia.

When an outcome-linked cohort becomes available, use these features inside the frozen M0 → M1 → M2 evaluation design.